In [ ]:
import kagglehub
import pandas as pd
pd.set_option('display.width', 2000)

import re
import itertools

# Download and open latest version of Musk Tweet Data
path = kagglehub.dataset_download("zjjc123/elon-musk-tweets-dataset")
print("Path to dataset files:", path)
df = pd.read_csv(path+"\\tweets.csv")
df['cleanText'] = df['Text'].str.lower().str.replace('-', ' ').str.replace('$','', regex=False).str.strip()
df['Date'] = pd.to_datetime(df['Date'])
print(f'Total number of Tweets Loaded: {len(df)}')

# Filter for tweets mentioning Tesla
categories_tsla = {'Name': ['TESLA'],
                   'Ticker': [a.lower() for a in ['TSLA']]}
categories_broader = {'Name': ['TESLA'],
                      'Ticker': [a.lower() for a in ['TSLA']],
                      'Product': [a.lower() for a in ["self driving","autopilot","fsd","full self-driving","model 3","model y","model s","model x","cybertruck","roadster","superfactory","super factory","SolarCity","cybercab","robovan","supercharger","megapack","CATL","LG","gigafactory"]],
                      'Union': [a.lower() for a in ["IG Metall", "IF Metall", "Transportarbetareförbundet", "Union", "Strike", "Pinkerton", "UAW", "NLRB","EEOC","FLRA","AFL-CIO","Labor"]],
                     }
categories_broadest = dict(categories_broader, **{'Political': [a.lower() for a in ['Republican', 'Liberal', 'Democrat', 'Repub', 'Democracy', 'Donald Trump', 'Joe Biden', 'Obama', 'Election', 'Congress', 'Senate', 'left wing', 'leftwing', 'right wing', 'rightwing' ,'woke']]})

# Small keyword select
keywords = [a for a in itertools.chain(*categories_tsla.values())] # A bit hacky but it works

df_tsla = df[df['cleanText'].str.contains('|'.join(keywords), case=False, na=False)]
df_tsla = df_tsla.sort_values('Date').reset_index(drop=True)
print(f"TSLA-relevant tweets: {len(df_tsla)}")

# Broader keyword selection
keywords = [a for a in itertools.chain(*categories_broader.values())]
df_broader = df[df['cleanText'].str.contains('|'.join(keywords), case=False, na=False)]
df_broader = df_broader.sort_values('Date').reset_index(drop=True)
print(f"Likely-Tesla-relevant tweets: {len(df_broader)}")

# For each dataframe we add categorical splits, providing basic text handling at this stage, later we may wish to use a more advanced encoding
for key, keywords in categories_tsla.items():
    df_tsla[key] = df_tsla['cleanText'].str.contains('|'.join(keywords), case=False, na=False)
for key, keywords in categories_broader.items():
    df_broader[key] = df_broader['cleanText'].str.contains('|'.join(keywords), case=False, na=False) 
for key, keywords in categories_broadest.items():
    df[key] = df['cleanText'].str.contains('|'.join(keywords), case=False, na=False) 


print('\n---------------------------------------------')
category_counts = df_tsla[categories_tsla.keys()].sum()
print("Tweets per Category for Tesla Relevant:")
print(str(category_counts).split('dtype')[0])
category_counts = df_broader[categories_broader.keys()].sum()
print("Tweets per Category for Likely-Tesla Relevant:")
print(str(category_counts).split('dtype')[0])
category_counts = df[categories_broadest.keys()].sum()
print("Tweets per Category for All Elon Musk Tweets:")
print(str(category_counts).split('dtype')[0])

Path to dataset files: C:\Users\osc16\.cache\kagglehub\datasets\zjjc123\elon-musk-tweets-dataset\versions\329
Total number of Tweets Loaded: 2587
TSLA-relevant tweets: 276
Likely-Tesla-relevant tweets: 333

---------------------------------------------
Tweets per Category for Tesla Relevant:
Name      276
Ticker      0

Tweets per Category for Likely-Tesla Relevant:
Name       276
Ticker       0
Product    112
Union        5

Tweets per Category for All Elon Musk Tweets:
Name         276
Ticker         0
Product      112
Union          5
Political     23

